## Fine-tuning Helsinki opus-mt for Arabic -> English, with BLEU / chrF++ / COMET + challenge inference

Fine-tunes `Helsinki-NLP/opus-mt-tc-big-ar-en` on `train.csv` (source = `ar`, target = `en`), evaluates the fine-tuned model on the dev set, then translates the Arabic challenge file into a single `en` column.

**On Kaggle:** enable the **GPU** accelerator and **internet** (the checkpoint, COMET, and the trainer's hub downloads all need it).

**Direction:** everything is **ar -> en**. The challenge file holds Arabic sentences; the deliverable is their English translation in a column named `en`.

**Pipeline:** train -> reload best (fp32) -> generate dev + challenge -> free MT model -> COMET-score the dev set.

## 1. Install & Imports

In [ ]:
# Light deps (no kernel restart needed)
!pip install -q sentencepiece sacremoses sacrebleu

In [ ]:
import subprocess, sys

def pip(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args.split(),
                   check=True)

# Pin a known-good COMET + dependency stack, and ensure accelerate for the Trainer
pip("unbabel-comet==2.2.2")
pip("datasets==2.19.0")
pip("fsspec==2024.3.1")
pip("accelerate")

print("Done -- now RESTART THE KERNEL before continuing.")

In [ ]:
import os, gc
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import sacrebleu
from datasets import Dataset, DatasetDict
from transformers import (
    MarianMTModel, MarianTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Config

In [ ]:
# ============================================================
# CONFIG
# ============================================================
class Config:
    # --- Model: Arabic -> English (the direction we fine-tune) ---
    MODEL_NAME  = "Helsinki-NLP/opus-mt-tc-big-ar-en"

    # --- COMET model (reference-based, multilingual) ---
    COMET_MODEL = "Unbabel/wmt22-comet-da"

    # --- Data paths (adjust to your Kaggle dataset mounts) ---
    TRAIN_CSV     = "/kaggle/input/your-dataset/train.csv"
    DEV_CSV       = "/kaggle/input/datasets/mishbhaul/english-arabic-devtest/devtest.csv"
    CHALLENGE_TXT = "/kaggle/input/your-dataset/test_challenge_ar_ar-to-en.txt"

    EN_COL = "en"          # English (target)
    AR_COL = "ar"          # Arabic  (source)
    SPLIT_COL  = "split"   # dev CSV split column; None to skip filtering
    SPLIT_VALUE = "devtest"

    # --- Training ---
    SEED          = 42
    VAL_SIZE      = 0.05   # held-out slice of train.csv for in-training monitoring
    EPOCHS        = 3
    LR            = 2e-5
    TRAIN_BS      = 8      # drop to 4 if you OOM on the T4
    EVAL_BS       = 8
    GRAD_ACCUM    = 2      # effective batch = TRAIN_BS * GRAD_ACCUM
    MAX_LEN_TRAIN = 128    # shorter cap = less memory + faster; raise if you truncate a lot

    # --- Inference / generation (used for dev + challenge) ---
    BATCH_SIZE  = 16
    NUM_BEAMS   = 4
    MAX_LENGTH  = 256
    COMET_BATCH = 16
    USE_FP16    = False    # Marian beam search can degrade in fp16; fp32 inference is safe

    # --- Output ---
    OUTPUT_DIR = "/kaggle/working"
    CKPT_DIR   = "/kaggle/working/ckpts"
    BEST_DIR   = "/kaggle/working/opus-ar-en-finetuned"


CFG = Config()

## 3. Data Loading

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================
def load_parallel_csv(path, en_col="en", ar_col="ar",
                      split_col=None, split_value=None):
    """Load aligned English/Arabic sentences from a single CSV.
    Returns (english_list, arabic_list)."""
    df = pd.read_csv(path, encoding="utf-8")
    missing = {en_col, ar_col} - set(df.columns)
    assert not missing, f"CSV missing columns: {missing}. Found: {list(df.columns)}"

    if split_col and split_col in df.columns and split_value is not None:
        df = df[df[split_col] == split_value]

    n0 = len(df)
    df = df.dropna(subset=[en_col, ar_col])
    df[en_col] = df[en_col].astype(str).str.strip()
    df[ar_col] = df[ar_col].astype(str).str.strip()
    df = df[(df[en_col] != "") & (df[ar_col] != "")]
    print(f"Loaded {len(df)} valid pairs from {path} (dropped {n0 - len(df)})")
    return df[en_col].tolist(), df[ar_col].tolist()


def build_datasets(path, cfg):
    """Read train.csv, clean, and split into train/validation HF Datasets."""
    df = pd.read_csv(path, encoding="utf-8")
    missing = {cfg.EN_COL, cfg.AR_COL} - set(df.columns)
    assert not missing, f"train CSV missing columns: {missing}. Found: {list(df.columns)}"

    n0 = len(df)
    df = df.dropna(subset=[cfg.EN_COL, cfg.AR_COL])
    df[cfg.EN_COL] = df[cfg.EN_COL].astype(str).str.strip()
    df[cfg.AR_COL] = df[cfg.AR_COL].astype(str).str.strip()
    df = df[(df[cfg.EN_COL] != "") & (df[cfg.AR_COL] != "")]
    print(f"Training pairs: {len(df)} (dropped {n0 - len(df)})")

    ds = Dataset.from_pandas(df[[cfg.AR_COL, cfg.EN_COL]], preserve_index=False)
    split = ds.train_test_split(test_size=cfg.VAL_SIZE, seed=cfg.SEED)
    dd = DatasetDict(train=split["train"], validation=split["test"])
    print(dd)
    return dd


def load_challenge_txt(path):
    """One Arabic sentence per line. Keeps EVERY line (incl. blanks) so the
    output row count/order matches the input exactly -- critical for submissions."""
    with open(path, encoding="utf-8") as f:
        lines = [ln.strip() for ln in f.read().splitlines()]
    n_empty = sum(1 for l in lines if not l)
    print(f"Loaded {len(lines)} challenge lines from {path} ({n_empty} empty)")
    return lines

## 4. Model, COMET & Tokenization

In [ ]:
# ============================================================
# MODEL LOADING + TOKENIZATION HELPERS
# ============================================================
def load_base_model(model_name):
    """Load the base opus-mt checkpoint for fine-tuning (fp32 master weights)."""
    print(f"Loading base model {model_name} ...")
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(DEVICE)
    print("  loaded.")
    return tokenizer, model


def load_trained(best_dir):
    """Reload the fine-tuned model in fp32 for clean, stable beam-search inference."""
    print(f"Reloading fine-tuned model (fp32) from {best_dir} ...")
    tok = MarianTokenizer.from_pretrained(best_dir)
    mdl = MarianMTModel.from_pretrained(best_dir, dtype=torch.float32).to(DEVICE)
    mdl.eval()
    return tok, mdl


def load_comet(comet_model):
    from comet import download_model, load_from_checkpoint
    print(f"Loading COMET: {comet_model} ...")
    ckpt = download_model(comet_model)
    model = load_from_checkpoint(ckpt)
    print("COMET loaded.")
    return model


def free_model(model):
    """Release a model from GPU memory (we juggle the MT model + COMET on one T4)."""
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def make_tokenize_fn(tokenizer, cfg):
    """Source = Arabic (inputs), target = English (labels)  ->  ar2en."""
    def fn(batch):
        return tokenizer(
            batch[cfg.AR_COL],
            text_target=batch[cfg.EN_COL],
            max_length=cfg.MAX_LEN_TRAIN,
            truncation=True,
        )
    return fn


def make_compute_metrics(tokenizer):
    """sacreBLEU + chrF++ on the held-out slice for in-training monitoring.
    (Final dev numbers come from the fp32 reload + beam search below.)"""
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        dec_preds = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
        dec_refs  = [r.strip() for r in tokenizer.batch_decode(labels, skip_special_tokens=True)]
        bleu   = sacrebleu.corpus_bleu(dec_preds, [dec_refs]).score
        chrfpp = sacrebleu.corpus_chrf(dec_preds, [dec_refs], word_order=2).score
        return {"bleu": round(bleu, 2), "chrf++": round(chrfpp, 2)}
    return compute_metrics

## 5. Training

In [ ]:
# ============================================================
# TRAINING  (Seq2SeqTrainer, ar -> en)
# ============================================================
def train_model(model, tokenizer, tokenized, cfg):
    collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    args = Seq2SeqTrainingArguments(
        output_dir=cfg.CKPT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        learning_rate=cfg.LR,
        per_device_train_batch_size=cfg.TRAIN_BS,
        per_device_eval_batch_size=cfg.EVAL_BS,
        gradient_accumulation_steps=cfg.GRAD_ACCUM,
        weight_decay=0.01,
        warmup_ratio=0.1,
        num_train_epochs=cfg.EPOCHS,
        predict_with_generate=True,
        generation_max_length=cfg.MAX_LEN_TRAIN,
        fp16=(DEVICE == "cuda"),
        group_by_length=True,          # bucket similar lengths -> less padding
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        report_to="none",
        seed=cfg.SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=collator,
        processing_class=tokenizer,    # transformers>=4.46; use tokenizer= on older versions
        compute_metrics=make_compute_metrics(tokenizer),
    )

    print("Starting fine-tuning (ar -> en) ...")
    trainer.train()

    trainer.save_model(cfg.BEST_DIR)
    tokenizer.save_pretrained(cfg.BEST_DIR)
    print(f"Saved fine-tuned model -> {cfg.BEST_DIR}")
    return trainer

## 6. Translation

In [ ]:
# ============================================================
# TRANSLATION  (Helsinki / MarianMT)
# ============================================================
@torch.no_grad()
def translate(sentences, tokenizer, model, cfg):
    """Batch-translate ar -> en with the fine-tuned model."""
    outputs = []
    for i in range(0, len(sentences), cfg.BATCH_SIZE):
        batch = sentences[i : i + cfg.BATCH_SIZE]
        enc = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=cfg.MAX_LENGTH,
        ).to(DEVICE)
        gen = model.generate(
            **enc,
            num_beams=cfg.NUM_BEAMS,
            max_length=cfg.MAX_LENGTH,
            early_stopping=True,
        )
        outputs.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
        done = min(i + cfg.BATCH_SIZE, len(sentences))
        print(f"  {done}/{len(sentences)}", end="\r")
    print()
    return outputs


def translate_preserve_order(lines, tokenizer, model, cfg):
    """Translate while keeping output count/order EXACTLY equal to input.
    Empty input lines map to empty output (so submission rows stay aligned)."""
    idx = [i for i, l in enumerate(lines) if l.strip()]
    src = [lines[i] for i in idx]
    hyp = translate(src, tokenizer, model, cfg) if src else []
    full = [""] * len(lines)
    for i, t in zip(idx, hyp):
        full[i] = t
    return full

## 7. Evaluation Metrics

In [ ]:
# ============================================================
# EVALUATION
# ============================================================
def eval_surface(hypotheses, references):
    """sacreBLEU BLEU + chrF++ (chrF with word_order=2)."""
    bleu   = sacrebleu.corpus_bleu(hypotheses, [references])
    chrfpp = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)
    return bleu.score, chrfpp.score


def eval_comet(sources, hypotheses, references, comet_model, cfg):
    """Reference-based COMET score (0-1, higher better)."""
    data = [{"src": s, "mt": h, "ref": r}
            for s, h, r in zip(sources, hypotheses, references)]
    out = comet_model.predict(
        data, batch_size=cfg.COMET_BATCH,
        gpus=1 if DEVICE == "cuda" else 0,
    )
    return out["system_score"], out["scores"]


def evaluate(sources, hypotheses, references, direction_name, comet_model, cfg):
    """Run all three metrics for one direction."""
    bleu, chrfpp = eval_surface(hypotheses, references)
    comet_sys, comet_each = eval_comet(sources, hypotheses, references, comet_model, cfg)

    print(f"\n=== {direction_name} ===")
    print(f"BLEU   : {bleu:.2f}")
    print(f"chrF++ : {chrfpp:.2f}")
    print(f"COMET  : {comet_sys:.4f}")

    return {
        "direction": direction_name,
        "bleu": round(bleu, 2),
        "chrf++": round(chrfpp, 2),
        "comet": round(comet_sys, 4),
    }, comet_each


def save_outputs(src, hyp, ref, comet_each, tag, cfg):
    """Per-sentence dump with COMET score for error inspection."""
    df = pd.DataFrame({
        "source": src, "hypothesis": hyp, "reference": ref,
        "comet": [round(c, 4) for c in comet_each],
    })
    path = f"{cfg.OUTPUT_DIR}/{tag}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved outputs -> {path}")
    return df

## 8. Main

In [ ]:
# ============================================================
# MAIN
# ============================================================
def run():
    # ---- 1. Build + tokenize training data (ar -> en) ----
    tokenizer, base = load_base_model(CFG.MODEL_NAME)
    ds = build_datasets(CFG.TRAIN_CSV, CFG)
    tokenized = ds.map(
        make_tokenize_fn(tokenizer, CFG),
        batched=True,
        remove_columns=ds["train"].column_names,
    )

    # ---- 2. Fine-tune ----
    trainer = train_model(base, tokenizer, tokenized, CFG)
    del trainer, base
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # ---- 3. Reload best model (fp32) for inference ----
    tok, model = load_trained(CFG.BEST_DIR)

    # ---- 4. Generate everything while the MT model is on GPU ----
    english, arabic = load_parallel_csv(
        CFG.DEV_CSV, CFG.EN_COL, CFG.AR_COL, CFG.SPLIT_COL, CFG.SPLIT_VALUE
    )
    print("Translating dev set (ar -> en) ...")
    dev_hyp = translate(arabic, tok, model, CFG)

    print("Translating challenge set (ar -> en) ...")
    challenge_ar = load_challenge_txt(CFG.CHALLENGE_TXT)
    challenge_en = translate_preserve_order(challenge_ar, tok, model, CFG)

    free_model(model)   # MT generation done; free VRAM before COMET

    # ---- 5. Score the dev set (BLEU / chrF++ / COMET) ----
    comet = load_comet(CFG.COMET_MODEL)
    row, comet_each = evaluate(
        arabic, dev_hyp, english, "Arabic -> English (fine-tuned)", comet, CFG
    )
    save_outputs(arabic, dev_hyp, english, comet_each, "dev_ar2en_finetuned", CFG)

    summary = pd.DataFrame([row])
    summary.to_csv(f"{CFG.OUTPUT_DIR}/finetuned_dev_summary.csv", index=False)
    print("\n" + "=" * 45)
    print(summary.to_string(index=False))

    # ---- 6. Save challenge translations: ONLY the 'en' column ----
    out = pd.DataFrame({"en": challenge_en})
    ch_path = f"{CFG.OUTPUT_DIR}/challenge_translations.csv"
    out.to_csv(ch_path, index=False, encoding="utf-8-sig")
    print(f"Saved challenge translations -> {ch_path} "
          f"(column: 'en', rows: {len(out)})")

    return summary


summary = run()